In [ ]:
import requests
import json
import time

API_URL = "https://api.crossref.org/works/"
RATE_LIMIT = 50
EMAIL_PARAM = "mailto:hogehoge@sample.com"
DOIs_file = "doi_list.txt"
OUTPUT_FILE = "authors_metadata.tsv"

def fetch_author_info(doi):
    headers = {
        "User-Agent": EMAIL_PARAM
    }

    url = f"{API_URL}{doi}"
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        data = response.json()
        authors = data.get("message", {}).get("author", [])

        author_info = []
        for i, author in enumerate(authors, start=1):
            orcid = author.get("ORCID", "")
            given_name = author.get("given", "")
            family_name = author.get("family", "")
            affiliation = ", ".join([aff.get("name", "") for aff in author.get("affiliation", [])])
            sequence = author.get("sequence", "")

            # Convert ORCID URLs from http to https if present
            orcid = orcid.replace("http://", "https://")

            author_info.append((orcid, f"{family_name}, {given_name}", affiliation, sequence))

        return author_info
    else:
        print(f"Failed to fetch data for DOI: {doi}, Status code: {response.status_code}")
        return []

def main():
    with open(DOIs_file, "r") as f:
        dois = f.read().splitlines()

    with open(OUTPUT_FILE, "w") as output_file:
        output_file.write("DOI queue number\tDOI\tAuthor queue number\tORCID\tName\tAffiliation\tSequence\n")

        total_processed = 0
        start_time = time.time()

        for i, doi in enumerate(dois, start=1):
            author_info = fetch_author_info(doi)

            for j, (orcid, name, affiliation, sequence) in enumerate(author_info, start=1):
                output_file.write(f"{i}\t{doi}\t{j}\t{orcid}\t{name}\t{affiliation}\t{sequence}\n")

            total_processed += len(author_info)
            time_elapsed = time.time() - start_time

            if total_processed % RATE_LIMIT == 0 and i < len(dois):
                time.sleep(1)

        print(f"Processing completed. Processed {total_processed} entries in {time_elapsed:.2f} seconds.")

if __name__ == "__main__":
    main()
